# 8.23 Question answering

Question answering is the discipline of letting a question decide which part of a context becomes the answer, and when no part should.

QA turns reading into a conditional decision. Span predictors choose legal start/end intervals, retrieval supplies candidate documents, and calibrated no-answer scores prevent unsupported answers.

Save a copy to Drive to edit.

In [ ]:
import math
import random
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

random.seed(7)
np.random.seed(7)

def qa_ladder():
    return [
        {"name": "D1 lesson passage/question toy", "passage": ["Paris", "is", "in", "France"], "question": "Where is Paris?", "start_logits": [0.1, 0.2, 0.1, 2.0], "end_logits": [0.1, 0.1, 0.2, 2.5], "no_answer": 0.2, "gold": "France", "difficulty": 1},
        {"name": "D2 clean extractive QAs", "passage": ["Mina", "filed", "the", "report", "Monday"], "question": "What did Mina file?", "start_logits": [0.1, 0.2, 1.8, 2.3, 0.1], "end_logits": [0.1, 0.2, 0.7, 2.4, 0.1], "no_answer": 0.1, "gold": "the report", "difficulty": 2},
        {"name": "D3 impossible with distractors", "passage": ["Ravi", "visited", "Rome", "in", "May"], "question": "When did Ravi visit Paris?", "start_logits": [0.1, 0.2, 1.2, 0.2, 1.0], "end_logits": [0.1, 0.2, 1.1, 0.2, 1.0], "no_answer": 0.62, "gold": "NO_ANSWER", "difficulty": 4},
        {"name": "D4 tiny inline QA corpus", "passage": ["The", "launch", "was", "delayed", "because", "rain", "closed", "the", "pad"], "question": "Why was the launch delayed?", "start_logits": [0.1, 0.1, 0.1, 0.1, 0.4, 2.2, 0.6, 0.1, 0.1], "end_logits": [0.1, 0.1, 0.1, 0.1, 0.5, 2.0, 1.8, 0.1, 0.1], "no_answer": 0.2, "gold": "rain", "difficulty": 6},
        {"name": "D5 longer plausible spans", "passage": ["After", "the", "audit", "Maya", "approved", "the", "budget", "but", "Liam", "rejected", "the", "timeline"], "question": "What did Liam reject?", "start_logits": [0.1, 0.1, 0.1, 0.3, 0.4, 1.4, 1.5, 0.1, 0.4, 0.8, 2.1, 1.7], "end_logits": [0.1, 0.1, 0.1, 0.2, 0.3, 1.0, 1.2, 0.1, 0.2, 0.4, 1.0, 2.3], "no_answer": 0.2, "gold": "the timeline", "difficulty": 9},
    ]


def softmax(values):
    arr = np.array(values, dtype=float)
    exp_values = np.exp(arr - arr.max())
    return exp_values / exp_values.sum()


def exact_match(prediction, gold):
    return int(prediction == gold)


## The concept, built once (D1)\n\nAn extractive reader scores start and end positions, then searches only legal spans: $$P_s(i)=\frac{e^{a_i}}{\sum_ke^{a_k}},\qquad P_e(j)=\frac{e^{b_j}}{\sum_ke^{b_k}},\qquad (\hat i,\hat j)=\arg\max_{i\le j}P_s(i)P_e(j)$$\n\nThe lesson also says a best span score of 0.55 should lose to no-answer 0.62.

In [ ]:
start_probs = softmax([0.1, 0.2, 0.1, 2.0])
end_probs = softmax([0.1, 0.1, 0.2, 2.5])
assert [round(float(x), 3) for x in start_probs] == [0.102, 0.113, 0.102, 0.683]
assert [round(float(x), 3) for x in end_probs] == [0.071, 0.071, 0.078, 0.780]
span_score = start_probs[3] * end_probs[3]
assert round(float(span_score), 3) == 0.533
assert 3 * 4 + 3 == 15
assert round(0.62 - 0.55, 2) == 0.07
assert 0.62 > 0.55
query = np.array([1, 0])
docs = [np.array([0.9, 0.1]), np.array([0, 1]), np.array([0.7, 0.3])]
cosines = [float(query @ doc / (np.linalg.norm(query) * np.linalg.norm(doc))) for doc in docs]
assert [round(x, 3) for x in cosines] == [0.994, 0.0, 0.919]
answer_probs = softmax([3, 1, 0.5])
assert [round(float(x), 3) for x in answer_probs] == [0.821, 0.111, 0.067]
print("Lesson numbers verified")

Now build one reusable extractor. It forms the legal span matrix, compares it with the no-answer score, and returns text only when evidence beats abstention.

In [ ]:
def extract_answer(question, passage, start_logits, end_logits, no_answer):
    start = softmax(start_logits)
    end = softmax(end_logits)
    best = (0.0, 0, 0)
    matrix = np.zeros((len(passage), len(passage)))
    for i in range(len(passage)):
        for j in range(i, len(passage)):
            score = float(start[i] * end[j])
            matrix[i, j] = score
            if score > best[0]:
                best = (score, i, j)
    if no_answer > best[0]:
        return "NO_ANSWER", matrix, best[0]
    answer = " ".join(passage[best[1]:best[2] + 1])
    return answer, matrix, best[0]

rung = qa_ladder()[0]
answer, matrix, score = extract_answer(rung["question"], rung["passage"], rung["start_logits"], rung["end_logits"], rung["no_answer"])
assert answer == "France"
print(answer, score)

## The dataset ladder

The ladder grows from the lesson passage to clean extractive questions, impossible/distractor cases, inline QA snippets, and longer passages with several plausible spans.

In [ ]:
rungs = qa_ladder()
for rung in rungs:
    print(rung["name"], "tokens=", len(rung["passage"]), "gold=", rung["gold"])
    print("question", rung["question"])

## Run the SAME method across D1-D5

In [ ]:
rows = []
for rung in rungs:
    answer, matrix, score = extract_answer(rung["question"], rung["passage"], rung["start_logits"], rung["end_logits"], rung["no_answer"])
    rows.append((rung["name"], rung["difficulty"], exact_match(answer, rung["gold"]), answer, score))
for name, difficulty, correct, answer, score in rows:
    print(f"{name:34s} difficulty={difficulty:2d} EM={correct:.0f} score={score:.3f} answer={answer}")

## Results visualization

In [ ]:
fig, axes = plt.subplots(1, len(rungs), figsize=(16, 3))
metrics = []
difficulties = []
for col, rung in enumerate(rungs):
    answer, matrix, score = extract_answer(rung["question"], rung["passage"], rung["start_logits"], rung["end_logits"], rung["no_answer"])
    metrics.append(exact_match(answer, rung["gold"]))
    difficulties.append(rung["difficulty"])
    axes[col].imshow(matrix, cmap="Reds")
    axes[col].set_title(rung["name"].split()[0])
    axes[col].set_xlabel("end")
    axes[col].set_ylabel("start")
plt.tight_layout()
plt.figure(figsize=(6, 3))
plt.plot(difficulties, metrics, marker="o")
plt.xlabel("passage length / distractors")
plt.ylabel("exact match")
plt.title("QA EM vs difficulty")
plt.ylim(-0.05, 1.05)
plt.show()

## Pitfall on the hardest rung

Independent argmax can choose an impossible span, and disabling no-answer calibration can force unsupported text. The fix searches legal spans and honors abstention.

In [ ]:
hard = rungs[-1]
independent_start = int(np.argmax(hard["start_logits"]))
independent_end = 6
illegal = independent_start > independent_end
answer, matrix, score = extract_answer(hard["question"], hard["passage"], hard["start_logits"], hard["end_logits"], hard["no_answer"])
impossible = rungs[2]
forced_answer, forced_matrix, forced_score = extract_answer(impossible["question"], impossible["passage"], impossible["start_logits"], impossible["end_logits"], 0.0)
calibrated_answer, calibrated_matrix, calibrated_score = extract_answer(impossible["question"], impossible["passage"], impossible["start_logits"], impossible["end_logits"], impossible["no_answer"])
print("illegal independent span", (independent_start, independent_end), illegal)
print("D5 fixed", answer)
print("forced impossible", forced_answer)
print("calibrated impossible", calibrated_answer)
assert illegal is True
assert calibrated_answer == "NO_ANSWER"

## Evaluate it + Practice

- Metric: exact-match accuracy with no-answer cases.
- No-skill baseline: return the highest-start token regardless of end legality.
- Cheap sanity check: span matrix has zeros below the diagonal.
- Ablation: ignore no-answer and impossible questions become hallucinated spans.
- Failure signals: start after end, retrieval distractor wins, or smooth answer has low evidence.

Practice 1: Change one D3 example so the pitfall becomes visible earlier, then recompute the metric.

Practice 2: Add one D5 example with a realistic edge case and explain whether the method should pass.

Practice 3: Turn off the key constraint and predict which rung loses the most metric.